# 03 — RAG Question Answering

## Responsible AI RAG Assistant

This notebook builds the first Retrieval-Augmented Generation question-answering layer for the Responsible AI RAG Assistant project.

The previous notebook created:

- local embeddings
- a persistent Chroma vector database
- semantic retrieval tests
- retrieval quality notes

This notebook will load the existing Chroma vector store, retrieve relevant chunks for user questions, format retrieved context, and create grounded answer structures.

## Notebook Goals

By the end of this notebook, we want to:

1. Load the existing Chroma vector database.
2. Load the same embedding model used in Notebook 02.
3. Retrieve relevant chunks for a user question.
4. Format retrieved chunks as source-grounded context.
5. Create a responsible RAG answer template.
6. Add basic guardrails and disclaimers.
7. Test the assistant with AI Act questions.

## Important Responsible-Use Note

This assistant is an educational and portfolio-ready prototype.

It does not provide legal advice.

Answers should be grounded in retrieved public-source context. If the retrieved context is insufficient, the assistant should say that the available sources do not support a confident answer.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import date

print("Notebook 03 setup started.")
print(f"Current date: {date.today()}")

Notebook 03 setup started.
Current date: 2026-07-02


In [2]:
# Project paths

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
VECTOR_STORE_DIR = DATA_DIR / "vector_store"
REPORTS_DIR = PROJECT_ROOT / "reports"

paths = {
    "PROJECT_ROOT": PROJECT_ROOT,
    "DATA_DIR": DATA_DIR,
    "PROCESSED_DATA_DIR": PROCESSED_DATA_DIR,
    "VECTOR_STORE_DIR": VECTOR_STORE_DIR,
    "REPORTS_DIR": REPORTS_DIR,
}

for name, path in paths.items():
    print(f"{name}: {path}")
    print(f"Exists: {path.exists()}")
    print("-" * 80)

PROJECT_ROOT: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant
Exists: True
--------------------------------------------------------------------------------
DATA_DIR: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data
Exists: True
--------------------------------------------------------------------------------
PROCESSED_DATA_DIR: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\processed
Exists: True
--------------------------------------------------------------------------------
VECTOR_STORE_DIR: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\vector_store
Exists: True
--------------------------------------------------------------------------------
REPORTS_DIR: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\reports
Exists: True
--------------------------------------------------------------------------------


## Load Retrieval Components

The RAG pipeline needs two retrieval components:

1. The same embedding model used in Notebook 02.
2. The existing Chroma vector collection created in Notebook 02.

The embedding model converts the user question into a vector.

Chroma then compares the question vector with stored chunk vectors and returns the most relevant document chunks.

In [3]:
from sentence_transformers import SentenceTransformer
import chromadb

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
collection_name = "responsible_ai_rag_src_001"

print(f"Loading embedding model: {embedding_model_name}")
embedding_model = SentenceTransformer(embedding_model_name)
print("Embedding model loaded.")

chroma_client = chromadb.PersistentClient(path=str(VECTOR_STORE_DIR))
collection = chroma_client.get_collection(name=collection_name)

print(f"Loaded Chroma collection: {collection_name}")
print(f"Collection count: {collection.count()}")

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.
Loaded Chroma collection: responsible_ai_rag_src_001
Collection count: 11


In [4]:
def retrieve_relevant_chunks(query: str, n_results: int = 4) -> pd.DataFrame:
    """
    Retrieve relevant chunks from the Chroma vector store for a user query.
    
    Parameters
    ----------
    query : str
        User question.
    n_results : int
        Number of chunks to retrieve.
    
    Returns
    -------
    pd.DataFrame
        Retrieved chunks with metadata and distance scores.
    """
    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    ).astype(float).tolist()
    
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["documents", "metadatas", "distances"],
    )
    
    records = []
    
    for rank, (document, metadata, distance) in enumerate(
        zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        ),
        start=1,
    ):
        records.append(
            {
                "rank": rank,
                "distance": distance,
                "source_id": metadata["source_id"],
                "source_name": metadata["source_name"],
                "publisher": metadata["publisher"],
                "source_type": metadata["source_type"],
                "url": metadata["url"],
                "chunk_index": metadata["chunk_index"],
                "word_count": metadata["word_count"],
                "retrieved_text": document,
            }
        )
    
    return pd.DataFrame(records)


print("Retrieval function is ready.")

Retrieval function is ready.


In [5]:
test_question = "What is the risk-based approach of the EU AI Act?"

retrieved_chunks_df = retrieve_relevant_chunks(test_question, n_results=4)

print(f"Question: {test_question}")
print(f"Retrieved chunks: {len(retrieved_chunks_df)}")

retrieved_chunks_df[
    [
        "rank",
        "distance",
        "source_id",
        "source_name",
        "chunk_index",
        "word_count",
        "retrieved_text",
    ]
]

Question: What is the risk-based approach of the EU AI Act?
Retrieved chunks: 4


,rank,distance,source_id,source_name,chunk_index,word_count,retrieved_text
0,1,0.404169,SRC-001,AI Act,2,250,"of the AI Act ahead of time. In parallel, the ..."
1,2,0.537072,SRC-001,AI Act,1,250,The AI Act is the first-ever legal framework o...
2,3,0.711476,SRC-001,AI Act,10,250,"on general-purpose AI models, reducing governa..."
3,4,0.721542,SRC-001,AI Act,8,250,deepfakes) as well as text. The Guidelines on ...


## Format Retrieved Context

A RAG system should not simply pass raw retrieved chunks to the language model without structure.

The retrieved context should include:

- source ID
- source name
- publisher
- chunk index
- URL
- retrieved text

This makes the final answer easier to ground and easier to audit.

In [6]:
def format_retrieved_context(retrieved_df: pd.DataFrame, max_chars_per_chunk: int = 1200) -> str:
    """
    Format retrieved chunks into a structured context block.
    
    Parameters
    ----------
    retrieved_df : pd.DataFrame
        Retrieved chunks from the vector store.
    max_chars_per_chunk : int
        Maximum number of characters to include per chunk.
    
    Returns
    -------
    str
        Formatted context string.
    """
    if retrieved_df.empty:
        return "No retrieved context available."
    
    context_blocks = []
    
    for _, row in retrieved_df.iterrows():
        chunk_text = row["retrieved_text"][:max_chars_per_chunk]
        
        block = f"""
[Retrieved Chunk {row['rank']}]
Source ID: {row['source_id']}
Source Name: {row['source_name']}
Publisher: {row['publisher']}
Chunk Index: {row['chunk_index']}
URL: {row['url']}

Text:
{chunk_text}
""".strip()
        
        context_blocks.append(block)
    
    return "\n\n" + ("-" * 100 + "\n\n").join(context_blocks)


formatted_context = format_retrieved_context(retrieved_chunks_df)

print(formatted_context[:3000])



[Retrieved Chunk 1]
Source ID: SRC-001
Source Name: AI Act
Publisher: European Commission
Chunk Index: 2
URL: https://digital-strategy.ec.europa.eu/en/policies/regulatory-framework-ai

Text:
of the AI Act ahead of time. In parallel, the AI Act Service Desk is also providing information and support for a smooth and effective implementation of the AI Act across the EU. Why do we need rules on AI? The AI Act ensures that Europeans can trust what AI has to offer. While most AI systems pose limited to no risk and can contribute to solving many societal challenges, certain AI systems create risks that we must address to avoid undesirable outcomes. For example, it is often not possible to find out why an AI system has made a decision or prediction and taken a particular action. So, it may become difficult to assess whether someone has been unfairly disadvantaged, such as in a hiring decision or in an application for a public benefit scheme. Although existing legislation provides some protec

## Build Responsible RAG Prompt

The retrieved context will now be inserted into a structured RAG prompt.

The prompt instructs the assistant to:

- answer only from retrieved context
- avoid legal advice
- avoid unsupported claims
- mention when the context is insufficient
- include a responsible-use note
- keep the answer source-aware

This prompt will later be used with an LLM, but first we inspect it manually.

In [7]:
def build_rag_prompt(question: str, retrieved_context: str) -> str:
    """
    Build a responsible RAG prompt using the user question and retrieved context.

    This prompt is designed for later use with an LLM.
    """
    prompt = f"""
You are a Responsible AI knowledge-support assistant.

You answer questions using only the retrieved context provided below.

Important rules:
1. Do not provide legal advice.
2. Do not claim certainty beyond the retrieved context.
3. If the retrieved context is insufficient, say that the available sources do not provide enough information.
4. Mention that real compliance decisions should be reviewed by qualified legal, privacy, or compliance professionals.
5. Keep the answer clear, structured, and source-aware.

User question:
{question}

Retrieved context:
{retrieved_context}

Write a grounded answer with:
- a direct answer
- key supporting points from the retrieved context
- source references using source ID and chunk index
- a short responsible-use note
""".strip()

    return prompt


rag_prompt = build_rag_prompt(test_question, formatted_context)

print(rag_prompt[:3500])

You are a Responsible AI knowledge-support assistant.

You answer questions using only the retrieved context provided below.

Important rules:
1. Do not provide legal advice.
2. Do not claim certainty beyond the retrieved context.
3. If the retrieved context is insufficient, say that the available sources do not provide enough information.
4. Mention that real compliance decisions should be reviewed by qualified legal, privacy, or compliance professionals.
5. Keep the answer clear, structured, and source-aware.

User question:
What is the risk-based approach of the EU AI Act?

Retrieved context:


[Retrieved Chunk 1]
Source ID: SRC-001
Source Name: AI Act
Publisher: European Commission
Chunk Index: 2
URL: https://digital-strategy.ec.europa.eu/en/policies/regulatory-framework-ai

Text:
of the AI Act ahead of time. In parallel, the AI Act Service Desk is also providing information and support for a smooth and effective implementation of the AI Act across the EU. Why do we need rules on A

## First Context-Only Answer Prototype

Before connecting an external LLM, the notebook creates a simple context-only answer prototype.

This function does not generate new legal conclusions.

It creates a structured answer using:

- the user question
- the retrieved chunks
- source IDs
- chunk indexes
- a responsible-use disclaimer

This is useful as a safe baseline before adding an LLM-based answer generator.

In [8]:
def generate_context_only_answer(question: str, retrieved_df: pd.DataFrame) -> str:
    """
    Generate a simple source-grounded answer structure without calling an external LLM.

    This is a safe baseline answer format for the RAG pipeline.
    """
    if retrieved_df.empty:
        return (
            "I could not find enough retrieved context to answer this question from the available sources.\n\n"
            "Responsible-use note: This assistant does not provide legal advice. For real compliance decisions, "
            "consult qualified legal, privacy, or compliance professionals."
        )

    top_sources = retrieved_df[["source_id", "source_name", "chunk_index", "url"]].copy()

    evidence_lines = []
    for _, row in retrieved_df.head(3).iterrows():
        short_text = row["retrieved_text"][:450].replace("\n", " ").strip()
        evidence_lines.append(
            f"- Source {row['source_id']}, chunk {row['chunk_index']}: {short_text}..."
        )

    source_reference_lines = []
    for _, row in top_sources.drop_duplicates().iterrows():
        source_reference_lines.append(
            f"- {row['source_id']} — {row['source_name']}, chunk {row['chunk_index']}: {row['url']}"
        )

    answer = f"""
Question:
{question}

Context-grounded answer:
Based on the retrieved source context, the answer should be treated as a source-grounded summary rather than legal advice. The retrieved chunks indicate that the EU AI Act addresses AI risks through a structured regulatory approach and includes references to risk levels, high-risk systems, prohibited practices, transparency, and implementation support.

Key retrieved evidence:
{chr(10).join(evidence_lines)}

Source references:
{chr(10).join(source_reference_lines)}

Responsible-use note:
This prototype does not provide legal advice. The answer is based only on the retrieved public-source context currently available in the vector store. Real compliance decisions should be reviewed by qualified legal, privacy, or compliance professionals.
""".strip()

    return answer


context_only_answer = generate_context_only_answer(test_question, retrieved_chunks_df)

print(context_only_answer)

Question:
What is the risk-based approach of the EU AI Act?

Context-grounded answer:
Based on the retrieved source context, the answer should be treated as a source-grounded summary rather than legal advice. The retrieved chunks indicate that the EU AI Act addresses AI risks through a structured regulatory approach and includes references to risk levels, high-risk systems, prohibited practices, transparency, and implementation support.

Key retrieved evidence:
- Source SRC-001, chunk 2: of the AI Act ahead of time. In parallel, the AI Act Service Desk is also providing information and support for a smooth and effective implementation of the AI Act across the EU. Why do we need rules on AI? The AI Act ensures that Europeans can trust what AI has to offer. While most AI systems pose limited to no risk and can contribute to solving many societal challenges, certain AI systems create risks that we must address to avoid undesirable o...
- Source SRC-001, chunk 1: The AI Act is the first-ev

## Improve Context-Only Answer Function

The first answer prototype confirmed that the RAG pipeline can return a source-grounded answer structure.

However, the first prototype included wording that was specific to the risk-based AI Act question.

The improved version below is more reusable.

It will:

- work with different questions
- avoid hard-coded conclusions
- show retrieval confidence
- include top retrieved evidence
- show source references
- state when the available context may be insufficient
- keep the legal-advice disclaimer

In [9]:
def classify_retrieval_confidence(
    retrieved_df: pd.DataFrame,
    strong_threshold: float = 0.65,
    weak_threshold: float = 0.90
) -> str:
    """
    Classify retrieval confidence based on the top retrieval distance.
    
    Lower distance generally means stronger semantic similarity.
    Thresholds are simple project-level heuristics for this prototype.
    """
    if retrieved_df.empty:
        return "insufficient"
    
    top_distance = float(retrieved_df.iloc[0]["distance"])
    
    if top_distance <= strong_threshold:
        return "moderate_to_good"
    elif top_distance <= weak_threshold:
        return "weak_to_moderate"
    else:
        return "insufficient"


def generate_context_only_answer_v2(
    question: str,
    retrieved_df: pd.DataFrame,
    max_evidence_chars: int = 500
) -> str:
    """
    Generate a reusable source-grounded answer structure without calling an external LLM.
    
    This function does not create legal conclusions.
    It summarizes what the retrieved chunks appear to support and exposes the evidence.
    """
    confidence = classify_retrieval_confidence(retrieved_df)
    
    if retrieved_df.empty or confidence == "insufficient":
        return f"""
Question:
{question}

Context-grounded answer:
The available retrieved context is not sufficient to answer this question confidently from the current vector store.

Retrieval confidence:
{confidence}

Responsible-use note:
This prototype does not provide legal advice. It only searches and summarizes available public-source context. For real compliance decisions, consult qualified legal, privacy, or compliance professionals.
""".strip()
    
    evidence_lines = []
    for _, row in retrieved_df.head(3).iterrows():
        short_text = str(row["retrieved_text"]).replace("\n", " ").strip()
        short_text = short_text[:max_evidence_chars]
        
        evidence_lines.append(
            f"- Source {row['source_id']}, chunk {row['chunk_index']} "
            f"(distance: {row['distance']:.4f}): {short_text}..."
        )
    
    source_reference_lines = []
    for _, row in retrieved_df[
        ["source_id", "source_name", "publisher", "chunk_index", "url"]
    ].drop_duplicates().iterrows():
        source_reference_lines.append(
            f"- {row['source_id']} — {row['source_name']} "
            f"({row['publisher']}), chunk {row['chunk_index']}: {row['url']}"
        )
    
    answer = f"""
Question:
{question}

Context-grounded answer:
The retrieved context contains potentially relevant information for this question. The answer below should be treated as a source-grounded evidence summary, not as legal advice.

Retrieval confidence:
{confidence}

Key retrieved evidence:
{chr(10).join(evidence_lines)}

Source references:
{chr(10).join(source_reference_lines)}

Responsible-use note:
This prototype does not provide legal advice. The answer is based only on the retrieved public-source context currently available in the vector store. Real compliance decisions should be reviewed by qualified legal, privacy, or compliance professionals.
""".strip()
    
    return answer


print("Improved context-only answer function is ready.")

Improved context-only answer function is ready.


In [10]:
context_only_answer_v2 = generate_context_only_answer_v2(
    question=test_question,
    retrieved_df=retrieved_chunks_df
)

print(context_only_answer_v2)

Question:
What is the risk-based approach of the EU AI Act?

Context-grounded answer:
The retrieved context contains potentially relevant information for this question. The answer below should be treated as a source-grounded evidence summary, not as legal advice.

Retrieval confidence:
moderate_to_good

Key retrieved evidence:
- Source SRC-001, chunk 2 (distance: 0.4042): of the AI Act ahead of time. In parallel, the AI Act Service Desk is also providing information and support for a smooth and effective implementation of the AI Act across the EU. Why do we need rules on AI? The AI Act ensures that Europeans can trust what AI has to offer. While most AI systems pose limited to no risk and can contribute to solving many societal challenges, certain AI systems create risks that we must address to avoid undesirable outcomes. For example, it is often not possible to ...
- Source SRC-001, chunk 1 (distance: 0.5371): The AI Act is the first-ever legal framework on AI, which addresses the ris

## Test Context-Only RAG Baseline

The improved context-only answer function will now be tested with three AI Act questions.

The purpose is to evaluate whether the retrieval and answer-formatting pipeline can:

- retrieve relevant context
- classify retrieval confidence
- produce source-grounded evidence summaries
- include source references
- avoid legal advice
- clearly state responsible-use limitations

This is the baseline RAG answer layer before adding an external LLM.

In [11]:
rag_test_questions = [
    {
        "test_id": "RAG-001",
        "question": "What is the risk-based approach of the EU AI Act?",
    },
    {
        "test_id": "RAG-002",
        "question": "What obligations apply to high-risk AI systems?",
    },
    {
        "test_id": "RAG-003",
        "question": "Does the AI Act require transparency for AI-generated content?",
    },
]

rag_test_questions_df = pd.DataFrame(rag_test_questions)
rag_test_questions_df

,test_id,question
0,RAG-001,What is the risk-based approach of the EU AI Act?
1,RAG-002,What obligations apply to high-risk AI systems?
2,RAG-003,Does the AI Act require transparency for AI-ge...


In [12]:
rag_test_results = []

for test_case in rag_test_questions:
    test_id = test_case["test_id"]
    question = test_case["question"]
    
    retrieved_df = retrieve_relevant_chunks(question, n_results=4)
    confidence = classify_retrieval_confidence(retrieved_df)
    answer = generate_context_only_answer_v2(question, retrieved_df)
    
    rag_test_results.append(
        {
            "test_id": test_id,
            "question": question,
            "retrieved_chunks": len(retrieved_df),
            "top_chunk_index": int(retrieved_df.iloc[0]["chunk_index"]) if not retrieved_df.empty else None,
            "top_distance": float(retrieved_df.iloc[0]["distance"]) if not retrieved_df.empty else None,
            "retrieval_confidence": confidence,
            "answer_preview": answer[:500],
            "full_answer": answer,
        }
    )

rag_test_results_df = pd.DataFrame(rag_test_results)

rag_test_results_df[
    [
        "test_id",
        "question",
        "retrieved_chunks",
        "top_chunk_index",
        "top_distance",
        "retrieval_confidence",
        "answer_preview",
    ]
]

,test_id,question,retrieved_chunks,top_chunk_index,top_distance,retrieval_confidence,answer_preview
0,RAG-001,What is the risk-based approach of the EU AI Act?,4,2,0.404169,moderate_to_good,Question:\nWhat is the risk-based approach of ...
1,RAG-002,What obligations apply to high-risk AI systems?,4,2,0.588744,moderate_to_good,Question:\nWhat obligations apply to high-risk...
2,RAG-003,Does the AI Act require transparency for AI-ge...,4,5,0.613211,moderate_to_good,Question:\nDoes the AI Act require transparenc...


In [13]:
for _, row in rag_test_results_df.iterrows():
    print("=" * 120)
    print(f"TEST ID: {row['test_id']}")
    print("=" * 120)
    print(row["full_answer"])
    print()

TEST ID: RAG-001
Question:
What is the risk-based approach of the EU AI Act?

Context-grounded answer:
The retrieved context contains potentially relevant information for this question. The answer below should be treated as a source-grounded evidence summary, not as legal advice.

Retrieval confidence:
moderate_to_good

Key retrieved evidence:
- Source SRC-001, chunk 2 (distance: 0.4042): of the AI Act ahead of time. In parallel, the AI Act Service Desk is also providing information and support for a smooth and effective implementation of the AI Act across the EU. Why do we need rules on AI? The AI Act ensures that Europeans can trust what AI has to offer. While most AI systems pose limited to no risk and can contribute to solving many societal challenges, certain AI systems create risks that we must address to avoid undesirable outcomes. For example, it is often not possible to ...
- Source SRC-001, chunk 1 (distance: 0.5371): The AI Act is the first-ever legal framework on AI, which 

In [14]:
rag_test_results_output_path = PROCESSED_DATA_DIR / "rag_baseline_test_results_src_001.csv"

rag_test_results_df.to_csv(
    rag_test_results_output_path,
    index=False,
    encoding="utf-8"
)

print(f"Saved RAG baseline test results to: {rag_test_results_output_path}")
print(f"File exists: {rag_test_results_output_path.exists()}")
print(f"Rows saved: {len(rag_test_results_df)}")

Saved RAG baseline test results to: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\processed\rag_baseline_test_results_src_001.csv
File exists: True
Rows saved: 3


## Notebook Summary

This notebook created the first question-answering layer for the Responsible AI RAG Assistant project.

Completed steps:

1. Loaded the existing Chroma vector store created in Notebook 02.
2. Loaded the same local embedding model used for retrieval.
3. Created a reusable retrieval function.
4. Retrieved relevant chunks for a user question.
5. Formatted retrieved chunks into structured source-grounded context.
6. Built a responsible RAG prompt template.
7. Created a first context-only answer prototype.
8. Improved the answer function to avoid hard-coded conclusions.
9. Added retrieval confidence classification.
10. Tested the context-only RAG baseline with three AI Act questions.
11. Saved the baseline RAG test results locally.

## Current Status

The project now has a working local RAG baseline:

- local source ingestion
- text cleaning and chunking
- local embeddings
- local Chroma vector store
- semantic retrieval
- context formatting
- source-grounded answer structure
- responsible-use disclaimer
- saved test results

## Current Limitation

The vector store currently contains only one source:

`SRC-001 — European Commission AI Act overview page`

Because of this, answer quality is limited by source coverage.

The next project step should add more authoritative public sources before adding external LLM answer generation.

In [15]:
# Final notebook checkpoint

notebook_03_checkpoint = {
    "embedding_model": embedding_model_name,
    "chroma_collection_name": collection_name,
    "chroma_collection_count": collection.count(),
    "rag_test_questions": len(rag_test_questions_df),
    "rag_test_results": len(rag_test_results_df),
    "rag_test_results_file_exists": rag_test_results_output_path.exists(),
    "retrieval_confidence_values": rag_test_results_df["retrieval_confidence"].unique().tolist(),
}

print("Notebook 03 final checkpoint completed.")
print(f"Embedding model: {notebook_03_checkpoint['embedding_model']}")
print(f"Chroma collection name: {notebook_03_checkpoint['chroma_collection_name']}")
print(f"Chroma collection count: {notebook_03_checkpoint['chroma_collection_count']}")
print(f"RAG test questions: {notebook_03_checkpoint['rag_test_questions']}")
print(f"RAG test results: {notebook_03_checkpoint['rag_test_results']}")
print(f"RAG test results file exists: {notebook_03_checkpoint['rag_test_results_file_exists']}")
print(f"Retrieval confidence values: {notebook_03_checkpoint['retrieval_confidence_values']}")

Notebook 03 final checkpoint completed.
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Chroma collection name: responsible_ai_rag_src_001
Chroma collection count: 11
RAG test questions: 3
RAG test results: 3
RAG test results file exists: True
Retrieval confidence values: ['moderate_to_good']


## Next Project Step

The next recommended step is source expansion.

Before adding external LLM answer generation, the project should collect and process more authoritative public sources:

1. `SRC-002` — Official EU AI Act legal text from EUR-Lex
2. `SRC-003` — Official GDPR legal text from EUR-Lex
3. `SRC-004` — NIST AI Risk Management Framework PDF
4. `SRC-005` — NIST AI RMF overview page

After adding more sources, the project can improve:

- retrieval coverage
- answer grounding
- citation quality
- evaluation quality
- responsible AI documentation

The next notebook should be:

`04_source_expansion_and_retrieval_evaluation.ipynb`